# 🤖 מחברת 5: בינה מלאכותית למדעי הרוח
## מבוא למדעי הרוח הדיגיטליים | אוניברסיטת אריאל | תשפ"ו
### פרופ' שי גורדין
---
**מה נלמד במחברת זו?**
AI (Artificial Intelligence) = בינה מלאכותית – היכולת של מחשבים לבצע משימות הדורשות "חשיבה".

**נושאים:**
1. מהי AI ומה רלוונטי למדעי הרוח?
2. LLMs – מודלי שפה גדולים (ChatGPT, Claude)
3. Prompt Engineering – כיצד לשאול AI בצורה יעילה
4. שימוש ב-API של Claude לניתוח טקסטים
5. סיווג טקסטים עם AI
6. שיקולים אתיים

---
## 🗺️ מדריך שימוש במחברת

| | |
|---|---|
| ⏱️ **זמן מוערך** | 🏫 בכיתה: ~60 דקות | 🏠 בבית: ~40 דקות |
| 💻 **היכן להריץ** | Google Colab |
| 🔑 **מפתח API** | אופציונלי — כל הקוד **עובד ללא API** (יש מצב סימולציה) |

### 🆓 שימוש חינמי ב-AI אמיתי — Google Gemini
המחברת כוללת אפשרות להשתמש ב-**Google Gemini חינמית** (ללא תשלום):  
1. עברו ל-[Google AI Studio](https://aistudio.google.com/apikey)  
2. לחצו **"Get API Key"** → **"Create API key"**  
3. העתיקו את המפתח  
4. הדביקו אותו בתא המפתח למטה  

> ⚠️ **אל תשתפו מפתח API עם אף אחד** — שמרו אותו רק לעצמכם!  
> ⚠️ **אל תכתבו מפתח API ישירות בקוד** — השתמשו בשיטה מאובטחת (ראו למטה)

---


## חלק א: AI ומדעי הרוח – מפגש מרתק
### הגל השלישי של AI: LLMs
**LLM** (Large Language Model) = מודל שפה גדול שאומן על כמויות עצומות של טקסט.
| מודל | יוצר | שנה | מאפיין |
|------|-------|-----|--------|
| **GPT-4** | OpenAI | 2023 | שוק מסחרי |
| **Claude** | Anthropic | 2023 | בטיחות-ממוקד |
| **Gemini** | Google | 2024 | מולטימודלי |
| **Llama** | Meta | 2023 | קוד פתוח |
### AI למדעי הרוח – אפשרויות:
- **תרגום** טקסטים עתיקים (ארמית, יוונית, לטינית)
- **סיכום** מסמכים ארוכים
- **סיווג** טקסטים לקטגוריות
- **זיהוי** דמויות, מקומות, תקופות
- **שאלות ותשובות** על מסמכים היסטוריים
- **ניתוח** שינויי שפה לאורך זמן
### מגבלות חשובות:
- **הזיות (Hallucinations)**: AI ממציא עובדות שאינן נכונות
- **Cutoff תאריך**: לא יודע אירועים חדשים
- **הטיות**: משקף הטיות בנתוני האימון
- **עלות**: שימוש ב-API עולה כסף
לקריאה: Bender et al. (2021). *On the Dangers of Stochastic Parrots*. FAccT.

In [ ]:
# התקנת ספריות
# google-generativeai  = Gemini חינמי ← עיקרי לשיעור
# anthropic / openai   = Claude / GPT  ← בתשלום (אופציונלי)
!pip install google-generativeai anthropic openai python-bidi -q

print("✅ ספריות AI הותקנו!")
print()
print("🆓 Gemini (חינמי):  https://aistudio.google.com/apikey")
print("💰 Claude (בתשלום): https://console.anthropic.com/")
print("💰 GPT   (בתשלום): https://platform.openai.com/")


In [ ]:
# יבוא ספריות
import os
import re
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display, HTML
import requests
import google.generativeai as genai

matplotlib.rcParams['axes.unicode_minus'] = False

# ============================================================
# הגדרת מפתח API
# ============================================================
# אפשרות א׳ (מומלצת לשיעור) – הדביקו את מפתח Gemini כאן:
GEMINI_API_KEY = ""   # ←  הדביקו כאן את המפתח שהורדתם מ-aistudio.google.com

# אפשרות ב׳ – מ-Environment Variable (מאובטח יותר לפרויקטים אמיתיים):
# GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ מפתח Gemini הוגדר!")
else:
    print("⚠️  לא הוגדר מפתח Gemini – הנייד ישתמש בתגובות מדומות.")
    print("   קבלו מפתח חינמי: https://aistudio.google.com/apikey")

print()
print("✅ ספריות יובאו!")
print()
print("⚠️  כלל ברזל: לעולם אל תשתפו מפתח API עם אחרים!")
print("⚠️  לעולם אל תעלו מפתחות ל-GitHub!")


## חלק ב: Prompt Engineering – אמנות השאלה

### מהו Prompt Engineering?
**Prompt** = ההנחיה שאתם שולחים ל-AI.  
**Prompt Engineering** = אמנות ניסוח הנחיות יעילות.

### עקרונות Prompt Engineering טוב:

| עיקרון | תיאור | דוגמה |
|--------|-------|--------|
| **ספציפיות** | הגדירו את המשימה בדיוק | "סכמו ב-3 משפטים" לעומת "סכמו" |
| **הקשר** | תנו רקע רלוונטי | "אתם ארכיאולוג המנתח..." |
| **דוגמאות** | הראו דוגמה לפלט רצוי | Few-shot prompting |
| **פורמט** | ציינו את הפורמט הרצוי | "הגיבו ב-JSON" / "ב-רשימה מנוקדת" |
| **תפקיד** | הגדירו תפקיד ל-AI | "אתה חוקר היסטוריה המתמחה ב..." |

### Chain of Thought:
בקשו מה-AI להסביר את ההיגיון שלו: "חשבו שלב אחר שלב לפני שתענו..."


In [ ]:
# ============================================================
# דוגמאות Prompt Engineering
# ============================================================

# -- פונקציה לדמיית קריאת API --
def simulate_llm_response(prompt, context=""):
    """
    פונקציה שמדמה תגובת LLM (ללא API אמיתי).
    
    בפרויקט אמיתי: תחליפו בקריאת API:
    
        import anthropic
        client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
        message = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        return message.content[0].text
    """
    # תגובות מדומות לדוגמה
    demo_responses = {
        'summarize': """
**סיכום (3 משפטים):**
תל מגידו הוא אתר ארכיאולוגי חשוב בצפון ישראל, שנחפר מהמאה ה-19.
הממצאים מציגים 26 שכבות התיישבות מהניאוליתי ועד לתקופה הפרסית.
האתר זוהה עם המגידו המקראית, אחד מהאתרים האסטרטגיים החשובים בארץ ישראל.
        """,
        'classify': """
{
  "קטגוריה": "ארכיאולוגיה",
  "תת_קטגוריה": "ארכיאולוגיה ביבשה",  
  "תקופה": "ברונזה וברזל",
  "ביטחון": 0.92,
  "הסבר": "הטקסט עוסק בחפירות ארכיאולוגיות ושכבות התיישבות"
}
        """,
        'translate': """
**תרגום לעברית מודרנית:**
"ובשנה השלישית למלכותו של יאשיהו מלך יהודה, 
הגיע נבוכדנאצר מלך בבל לירושלים ויצור עליה."
        """
    }
    
    if 'סכמ' in prompt or 'סיכום' in prompt:
        return demo_responses['summarize']
    elif 'סיווג' in prompt or 'classify' in prompt.lower():
        return demo_responses['classify']
    elif 'תרגם' in prompt or 'תרגום' in prompt:
        return demo_responses['translate']
    else:
        return f"[תגובה מדומה לפרומפט: {prompt[:50]}...]"


# -- דוגמאות Prompts --
print("📋 דוגמאות Prompt Engineering:")
print("=" * 60)

prompts = {
    "❌ Prompt גרוע": "ספר לי על ארכיאולוגיה",
    
    "✅ Prompt טוב": """אתה ארכיאולוג המתמחה בארץ ישראל.
סכם את הטקסט הבא ב-3 משפטים בעברית פשוטה:

'תל מגידו הוא אתר ארכיאולוגי...'

פורמט: כותרת מודגשת + 3 משפטים.""",

    "✅✅ Few-shot": """סווג טקסט לקטגוריה לפי הדוגמאות:

דוגמה 1: "חפירות בתל מגידו" -> {"קטגוריה": "ארכיאולוגיה"}
דוגמה 2: "דוד המלך ירושלים"  -> {"קטגוריה": "היסטוריה"}

סווג: "כלים דיגיטליים לניתוח טקסט" -> """
}

for title, prompt in prompts.items():
    print(f"\n{title}:")
    print(f"  Prompt: {prompt[:80]}...")
    response = simulate_llm_response(prompt)
    print(f"  תגובה: {response[:100].strip()}...")


In [ ]:
# ============================================================
# 🆓  קריאה ל-Gemini (גוגל – חינמי!)
# ============================================================

def call_gemini(prompt, model_name="gemini-flash-latest"):
    """
    שולח פרומפט ל-Gemini ומחזיר את תגובת המודל.

    אם לא הוגדר GEMINI_API_KEY – מחזיר תגובה מדומה (simulate_llm_response).

    פרמטרים:
        prompt      – הטקסט שרוצים לשלוח ל-AI
        model_name  – שם המודל (ברירת מחדל: gemini-flash-latest, חינמי)

    דוגמה:
        תגובה = call_gemini("סכם ב-3 משפטים: 'תל מגידו הוא...'")
        print(תגובה)
    """
    if not GEMINI_API_KEY:
        print("💡 אין מפתח API – משתמש בתגובה מדומה.")
        return simulate_llm_response(prompt)

    try:
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"❌ שגיאה: {e}")
        print("   משתמש בתגובה מדומה במקום.")
        return simulate_llm_response(prompt)


# -- בדיקת חיבור --
print("🔌 בודק חיבור ל-Gemini...")
test_response = call_gemini("ענה רק בעברית: מה 2+2?")
print(f"תשובת Gemini: {test_response.strip()[:120]}")
print()
print("✅ הפונקציה call_gemini() מוכנה לשימוש!")


## חלק ג: ניתוח טקסטים עם Gemini API

### ניתוח ממוקד-משימה
הפונקציה `analyze_with_gemini()` עוטפת את `call_gemini()` ומתאימה את הפרומפט לפי המשימה הנדרשת:

| משימה | מה ה-AI עושה |
|-------|-------------|
| `summarize` | סיכום ב-3 משפטים בעברית |
| `classify` | סיווג לקטגוריה + תקופה + ביטחון (JSON) |
| `ner` | חילוץ ישויות: אנשים, מקומות, תקופות (JSON) |
| `translate` | תרגום לעברית מודרנית |

> 💡 **ללא מפתח API** — הפונקציה מחזירה תגובה מדומה לצורך הדגמה, כך שהמחברת עובדת גם ללא חיבור לרשת.

In [ ]:
# ============================================================
# ניתוח טקסטים עם Gemini API
# ============================================================

TASK_PROMPTS = {
    'summarize': "סכם את הטקסט הבא ב-3 משפטים בעברית ברורה:\n\n{text}",
    'classify':  (
        "סווג את הטקסט הבא לאחת הקטגוריות: ארכיאולוגיה / היסטוריה / מדעי הרוח הדיגיטליים.\n"
        "החזר JSON בלבד עם השדות: קטגוריה, תקופה, ביטחון (0-1).\n\n{text}"
    ),
    'ner': (
        "חלץ מהטקסט הבא ישויות בשם וסווג אותן.\n"
        "החזר JSON בלבד עם המפתחות: אנשים, מקומות, תקופות, אירועים.\n\n{text}"
    ),
    'translate': "תרגם את הטקסט הבא לעברית מודרנית, תוך שמירה על הסגנון המקורי:\n\n{text}",
}

DEMO_RESPONSES = {
    'summarize': (
        "תל מגידו הוא אתר ארכיאולוגי חשוב בעמק יזרעאל עם 26 שכבות התיישבות. "
        "החפירות חשפו ממצאים מהתקופה הכנענית ועד הפרסית. "
        "האתר מזוהה עם מגידו המקראית ונכלל ברשימת אתרי מורשת אונסקו."
    ),
    'classify':  '{"קטגוריה": "ארכיאולוגיה", "תקופה": "ברונזה–ברזל", "ביטחון": 0.92}',
    'ner':       '{"אנשים": ["גוטליב שומכר"], "מקומות": ["תל מגידו", "עמק יזרעאל"], "תקופות": ["כנענית", "ברונזה"], "אירועים": []}',
    'translate': '"ויהי בימי המלך — ובאותה עת בא נבוכדנאצר וישב על ירושלים."',
}

def analyze_with_gemini(text, task):
    """
    ניתוח טקסט עם Gemini לפי משימה נתונה.

    פרמטרים:
        text : הטקסט לניתוח
        task : 'summarize' | 'classify' | 'ner' | 'translate'

    מחזיר: תגובת Gemini כ-string, או תגובה מדומה אם אין מפתח API.
    """
    prompt = TASK_PROMPTS.get(task, "נתח את הטקסט הבא:\n\n{text}").format(text=text)

    if GEMINI_API_KEY:
        return call_gemini(prompt)
    else:
        print("💡 אין מפתח API — משתמש בתגובה מדומה.")
        return DEMO_RESPONSES.get(task, f"[תגובה מדומה: {task}]")


# -- הדגמה --
sample_texts = [
    (
        "חפירות בתל מגידו חשפו 26 שכבות התיישבות מ-7000 שנה של היסטוריה. "
        "הממצאים כוללים ארמונות מהתקופה הכנענית, אורוות מהתקופה האיסורית, "
        "ותעלת מים מתוחכמת. האתר זוהה עם המגידו המקראית."
    ),
    (
        "דוד המלך כבש את ירושלים מהיבוסים וייסד אותה לבירת ממלכת ישראל. "
        "בנו שלמה בנה את בית המקדש הראשון, שחרב בידי נבוכדנאצר בשנת 586 לפנה״ס."
    ),
]

print("🤖 ניתוח טקסטים עם Gemini:")
print("=" * 60)

for i, text in enumerate(sample_texts, 1):
    print(f"\n📄 טקסט {i}: {text[:60]}...")
    for task in ['summarize', 'classify']:
        print(f"  📋 {task}:")
        response = analyze_with_gemini(text, task)
        print(f"  {response.strip()[:160]}")
    print("-" * 50)


## חלק ד: סיווג טקסטים עם AI
### גישות לסיווג:
| גישה | כלים | יתרון | חסרון |
|------|-------|--------|-------|
| **Zero-shot** | LLM (Claude/GPT) | ללא נתוני אימון | פחות מדויק |
| **Few-shot** | LLM + דוגמאות | מאוזן | דורש דוגמאות |
| **Fine-tuning** | BERT/DistilBERT | הכי מדויק | דורש נתונים + GPU |
| **כלאיים** | Rule-based + ML | מהיר, ניתן לפרש | מורכב לבנות |

In [ ]:
# ============================================================
# Pipeline סיווג טקסטים (Zero-shot + Few-shot)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# -- נתוני אימון לדוגמה --
TRAINING_DATA = [
    # ארכיאולוגיה
    ("חפירה בתל מגידו חשפה ממצאים מהתקופה הכנענית", "ארכיאולוגיה"),
    ("ממצאים ארכיאולוגיים כוללים חרסים ומטבעות עתיקות", "ארכיאולוגיה"),
    ("שכבות ההתיישבות מציגות תרבויות שונות לאורך אלפי שנים", "ארכיאולוגיה"),
    ("אתר ארכיאולוגי ים המלח חשף מגילות וכתבים עתיקים", "ארכיאולוגיה"),
    ("חפירת מצילה ברחוב ירושלים גילתה מקוואות מהתקופה ההרודיאנית", "ארכיאולוגיה"),
    ("ממצאי ברזל מהמאה עשר לפנה ספירה מוכיחים פעילות מתכתית", "ארכיאולוגיה"),
    
    # היסטוריה
    ("דוד המלך ייסד את ירושלים לבירת ממלכת ישראל", "היסטוריה"),
    ("חורבן הבית הראשון בידי הבבלים שינה את ההיסטוריה היהודית", "היסטוריה"),
    ("ממלכת חשמונאים שלטה ביהודה מהמאה השנייה לפנה ספירה", "היסטוריה"),
    ("גירוש יהודי ספרד בשנת ד אלפים ורנב השפיע על יהדות כולה", "היסטוריה"),
    ("עליות לארץ ישראל בראשית המאה העשרים שינו את הדמוגרפיה", "היסטוריה"),
    ("קום המדינה בתש ח היה נקודת מפנה בהיסטוריה היהודית", "היסטוריה"),
    
    # מדעי הרוח הדיגיטליים
    ("ניתוח קורפוסים גדולים דורש כלים חישוביים מתקדמים", "DH"),
    ("ויזואליזציה של נתונים ארכיאולוגיים ב-GIS מפשטת מחקר", "DH"),
    ("מידול נושאים חושף דפוסים נסתרים בטקסטים היסטוריים", "DH"),
    ("OCR מאפשר דיגיטציה של כתבי יד וטקסטים ישנים", "DH"),
    ("רשתות חברתיות של דמויות היסטוריות ניתנות לניתוח חישובי", "DH"),
    ("בינה מלאכותית מסייעת לפרשנות כתבות עתיקות", "DH"),
]

# פיצול לאימון ובדיקה
texts, labels = zip(*TRAINING_DATA)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

# -- מודל TF-IDF + Naive Bayes --
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer=lambda x: list(re.sub(r'[^\u05D0-\u05EA\s]', ' ', x).split()))),
    ('clf', MultinomialNB(alpha=0.1))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("📊 תוצאות הסיווג:")
print("=" * 55)
print(classification_report(y_test, y_pred, zero_division=0))

# -- בדיקת טקסטים חדשים --
print("\n🔍 בדיקת טקסטים חדשים:")
new_texts = [
    "ממצאים ארכיאולוגיים חדשים בגיא בן הינום",
    "ניתוח טקסטים עם Python ו-NLTK",
    "מרד המכבים נגד הסלאוקידים"
]

for text in new_texts:
    pred = pipeline.predict([text])[0]
    proba = pipeline.predict_proba([text])[0]
    classes = pipeline.classes_
    
    print(f"\n  '{text[:45]}...'")
    print(f"  -> {pred} ", end="")
    for cls, prob in zip(classes, proba):
        print(f"| {cls}: {prob:.2f}", end="")
    print()


## חלק ה: שיקולים אתיים – AI ומדעי הרוח
### שאלות אתיות מרכזיות:
**1. הטיות (Bias)**
- AI מאומן על טקסטים אנגלים -> הטיה מערבית
- ייצוג-חסר של תרבויות לא-מערביות
- מגדר, גזע, מעמד – כולם מיוצגים בהטיות האימון

**2. Hallucinations**
- AI ממציא עובדות שנשמעות אמיתיות
- **מסוכן** במיוחד בהיסטוריה ובארכיאולוגיה
- **חובה** לאמת כל עובדה מול מקורות ראשוניים

**3. שקיפות וסיוע**  
- כיצד לציין שימוש ב-AI במחקר אקדמי?
- מהי הגבול בין "עזרה" ל"מחקר שנעשה על ידי AI"?

**4. פרטיות**  
- שליחת מסמכים לAPI = שיתוף עם חברה חיצונית
- מסמכים ארכיוניים רגישים?

### המלצות לשימוש אחראי:
בדקו עובדות שAI מספק  
ציינו שימוש ב-AI בפרסומים אקדמיים  
השתמשו ב-AI כ"עוזר" ולא כ"מחבר"  
שמרו על שיפוט ביקורתי

In [ ]:
# ============================================================
# בדיקת עובדות ו-Hallucinations
# ============================================================

# מסד ידע בסיסי לאימות (Fact Checking)
KNOWN_FACTS = {
    'מגידו': {
        'מיקום': 'עמק יזרעאל',
        'תקופות': ['כנענית', 'ברונזה', 'ברזל'],
        'חוקרים': ['גוטליב שומכר', 'קלרנס פישר']
    },
    'ירושלים': {
        'ייסוד': 'לפחות 3500 שנה',
        'בניין_מקדש': 'שלמה המלך',
        'חורבן_ראשון': '586 לפנה״ס'
    },
    'גניזה קהירית': {
        'מיקום': 'קהיר, מצרים',
        'תקופה': 'המאה ה-9 עד ה-20',
        'גודל': 'כ-300,000 קטעים'
    }
}


def verify_claim(claim, fact_db=None):
    """
    בדיקת עובדה מול מסד ידע.
    
    בפרויקט אמיתי: ניתן לחבר ל-Wikidata API.
    """
    if fact_db is None:
        fact_db = KNOWN_FACTS
    
    warnings = []
    
    # חיפוש ישויות ידועות
    for entity, facts in fact_db.items():
        if entity in claim:
            # בדיקת עקביות
            for key, value in facts.items():
                if isinstance(value, str) and value in claim:
                    pass  # עובדה תואמת
    
    return warnings


# -- הדגמת זיהוי Hallucinations --
print("🔍 הדגמת בעיות Hallucinations:")
print("=" * 60)

ai_outputs = [
    {
        'prompt': 'מה ידוע על תל מגידו?',
        'response': 'תל מגידו הוא אתר ארכיאולוגי בעמק יזרעאל, שנחפר לראשונה ב-1903.',
        'correct': True,
        'note': 'נכון ✅'
    },
    {
        'prompt': 'מתי נחרב בית המקדש הראשון?',
        'response': 'בית המקדש הראשון נחרב בשנת 586 לפנה״ס בידי נבוכדנאצר מלך בבל.',
        'correct': True,
        'note': 'נכון ✅'
    },
    {
        'prompt': 'מי ביצע את חפירות הגניזה הקהירית?',
        'response': 'הגניזה הקהירית נחפרה בשנת 1890 על ידי סולומון שכטר ופרנקו מורטי.',
        'correct': False,
        'note': 'שגוי! ❌ פרנקו מורטי הוא חוקר ספרות, לא ארכיאולוג. הדמות הנוספת היא אחרת!'
    },
    {
        'prompt': 'מה גודל הגניזה הקהירית?',
        'response': 'הגניזה הקהירית מכילה כ-10,000 קטעי מסמכים.',
        'correct': False,
        'note': 'שגוי! ❌ הגניזה מכילה כ-300,000 קטעים – פי 30 מהתשובה!'
    }
]

for item in ai_outputs:
    status = "✅" if item['correct'] else "❌"
    print(f"\n{status} שאלה: {item['prompt']}")
    print(f"   תגובת AI: {item['response'][:80]}...")
    print(f"   ⚠️ {item['note']}")

print("\n🔑 מסקנה:")
print("   תמיד בדקו עובדות שAI מספק מול מקורות אמינים!")


In [ ]:
# ============================================================
# תהליך עבודה מעשי: AI + מחקר מדעי הרוח
# ============================================================

def humanities_research_pipeline(texts, research_question):
    """
    Pipeline מחקרי שמשלב AI עם מתודולוגיה הומניסטית.

    שלבים:
    1. קריאה מרחוק (כמות)
    2. AI סיכום וסיווג  ← Gemini אם הוגדר מפתח, אחרת תגובה מדומה
    3. זיהוי טקסטים מעניינים לקריאה קרובה
    4. קריאה קרובה אנושית

    פרמטרים:
        texts          : רשימת טקסטים
        research_question : שאלת המחקר


    מחזיר: DataFrame עם תוצאות
    """
    results = []

    print(f"🔬 שאלת מחקר: {research_question}")
    if GEMINI_API_KEY:
        print("🌐 מצב: API אמיתי (Gemini)")
    else:
        print("💡 מצב: תגובות מדומות — הגדירו GEMINI_API_KEY בתא 4 לתוצאות אמיתיות")
    print("=" * 60)

    for i, text in enumerate(texts, 1):
        print(f"\n📄 מסמך {i}/{len(texts)}: {text[:60]}...")

        # שלב 1: מטה-נתונים בסיסיים
        word_count = len(text.split())

        # שלב 2: AI – סיכום
        # משתמש ב-Gemini אם הוגדר מפתח, אחרת תגובה מדומה
        summary = analyze_with_gemini(text, 'summarize')
        classification = analyze_with_gemini(text, 'classify')

        # שלב 3: ציון "עניין"
        interesting_keywords = ['חשוב', 'ייחודי', 'ראשון', 'מרכזי', 'נדיר', 'חשף', 'גילה']
        interest_score = sum(1 for kw in interesting_keywords if kw in text)

        results.append({
            'מסמך':       i,
            'טקסט_קצר':   text[:80] + '...',
            'מילים':      word_count,
            'סיכום_AI':   summary.strip()[:200],
            'סיווג_AI':   classification.strip()[:100],
            'ציון_עניין': interest_score
        })

        print(f"   ✓ עובד | מילים: {word_count} | עניין: {interest_score}/7")

    df = pd.DataFrame(results)
    return df


# -- הרצה --
sample_for_pipeline = [
    # טקסט 1: ארכאולוגיה — תל מגידו
    """תל מגידו הוא אחד האתרים הארכאולוגיים החשובים ביותר בישראל ובעולם. החפירות שנערכו באתר מאז סוף המאה ה-19 חשפו כ-26 שכבות התיישבות רצופות, המשתרעות על פני כ-7,000 שנה של היסטוריה אנושית, מהתקופה הנאוליתית ועד לתקופה הפרסית. בין הממצאים הבולטים: ארמונות כנעניים מפוארים, אורוות מהתקופה האיסוראלית המיוחסות למלך אחאב, ותעלת מים מתוחכמת שנחצבה בסלע. האתר מזוהה עם מגידו המקראית, זירת קרבות מפורסמים לאורך ההיסטוריה, ונכלל ברשימת אתרי מורשת העולם של אונסקו.""",

    # טקסט 2: היסטוריה — גניזת קהיר
    """גניזת קהיר היא אוסף של כ-400,000 קטעי כתב יד יהודיים שנתגלו בסוף המאה ה-19 בבית הכנסת בן עזרא בפוסטאט (קהיר העתיקה). המסמכים, הכתובים בעיקר בערבית-יהודית, בעברית ובארמית, מכסים תקופה של כאלף שנה (המאה ה-9 עד ה-19 לספירה) ומספקים מידע רב-ערך על חיי היהודים בארצות האסלאם: מסחר, משפחה, דת, רפואה ושירה. המלומד שלמה דב גויטיין הקדיש את חייו לחקר הגניזה ופרסם את סדרת המחקרים המונומנטלית "חברה ים-תיכונית". כיום מאות אלפי הדפים ממשיכים להיסרק ולהיות נגישים לחוקרים ברחבי העולם דרך פרויקט Geniza@Princeton ומאגרים דיגיטליים נוספים.""",

    # טקסט 3: מדעי הרוח הדיגיטליים — ניתוח קורפוס
    """פרויקט השו"ת הוא מאגר דיגיטלי המכיל את כלל ספרות ההלכה היהודית — למעלה מ-300,000 שו"ת ומעל ל-3 מיליון עמודי טקסט — ומאפשר חיפוש טקסטואלי מלא. בשנים האחרונות שילבו חוקרים שיטות של עיבוד שפה טבעית (NLP) וקריאה מרחוק כדי לזהות תבניות היסטוריות בספרות זו: שינויים בשימוש במונחים לאורך הדורות, התפשטות פסיקות הלכתיות ממרכז אחד לאחר, ורשתות ציטוט בין פוסקים מתקופות שונות. גישה זו מאפשרת לשאול שאלות מחקריות חדשות שלא ניתן היה לענות עליהן בקריאה ידנית של אוסף בהיקף כזה.""",
]

df_results = humanities_research_pipeline(
    sample_for_pipeline,
    research_question="מהם הנושאים המרכזיים בכתיבה ארכיאולוגית-היסטורית עברית?"
)

print("\n📋 תוצאות ה-Pipeline:")
display(df_results[['מסמך', 'מילים', 'ציון_עניין', 'טקסט_קצר']])

df_results.to_csv('ai_analysis_results.csv', index=False, encoding='utf-8-sig')
print("\n💾 נשמר: ai_analysis_results.csv")


In [ ]:
# ============================================================
# ויזואליזציה של תוצאות AI
# ============================================================

from bidi.algorithm import get_display
import re, json as _json

def rtl(text):
    return get_display(str(text))

def extract_category(json_str):
    """חולץ קטגוריה — תומך גם ב-JSON עם גדרות markdown וגם בטקסט חופשי."""
    # נקה גדרות markdown: ```json ... ```
    clean = re.sub(r'```(?:json)?', '', str(json_str)).strip()
    m = re.search(r'"קטגוריה"\s*:\s*"([^"]+)"', clean)
    if m:
        return m.group(1).strip()
    # fallback: זיהוי לפי מילות מפתח
    if any(w in json_str for w in ['ארכיאולוג', 'חפירה', 'תל ', 'ממצא']):
        return 'ארכיאולוגיה'
    if any(w in json_str for w in ['היסטורי', 'מלך', 'חורבן', 'גניזה']):
        return 'היסטוריה'
    if any(w in json_str for w in ['דיגיטל', 'NLP', 'OCR', 'שו"ת', 'קורפוס']):
        return 'DH'
    return 'לא ידוע'

def extract_confidence(json_str):
    """חולץ ציון ביטחון — fallback ל-0.85."""
    clean = re.sub(r'```(?:json)?', '', str(json_str)).strip()
    m = re.search(r'"ביטחון"\s*:\s*([0-9.]+)', clean)
    return float(m.group(1)) if m else 0.85

def short_label(text):
    """2-3 מילים עבריות ראשונות כשם קצר."""
    words = [w for w in str(text).split() if re.search(r'[\u05d0-\u05ea]', w)]
    return ' '.join(words[:2])

# ── בניית DataFrame לויזואליזציה ────────────────────────────
try:
    df_viz = df_results.copy()
    source = "תוצאות Pipeline אמיתיות"
except NameError:
    try:
        df_viz = pd.read_csv('ai_analysis_results.csv')
        source = "קובץ ai_analysis_results.csv"
    except FileNotFoundError:
        df_viz = pd.DataFrame({
            'מסמך': [1, 2, 3],
            'טקסט_קצר': ['תל מגידו הוא אתר...', 'גניזת קהיר היא...', 'פרויקט השו"ת הוא...'],
            'מילים': [52, 38, 30],
            'סיווג_AI': ['{"קטגוריה": "ארכיאולוגיה", "ביטחון": 0.93}',
                         '{"קטגוריה": "היסטוריה", "ביטחון": 0.90}',
                         '{"קטגוריה": "DH", "ביטחון": 0.88}'],
        })
        source = "נתוני דוגמה"

df_viz['קטגוריה']     = df_viz['סיווג_AI'].apply(extract_category)
df_viz['ציון_ביטחון'] = df_viz['סיווג_AI'].apply(extract_confidence)
df_viz['שם_קצר']      = df_viz['טקסט_קצר'].apply(short_label)
df_viz['אורך_טקסט']   = df_viz['מילים']
print(f"📊 מקור הנתונים: {source}")
print(df_viz[['שם_קצר', 'קטגוריה', 'ציון_ביטחון', 'אורך_טקסט']].to_string(index=False))

# ── ציור ──────────────────────────────────────────────────────
colors = {'ארכיאולוגיה': '#e74c3c', 'היסטוריה': '#2980b9',
          'DH': '#27ae60', 'לא ידוע': '#888888'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# לוח 1: עוגה
cat_counts = df_viz['קטגוריה'].value_counts()
axes[0].pie(cat_counts,
            labels=[rtl(c) for c in cat_counts.index],
            colors=[colors.get(c, '#888') for c in cat_counts.index],
            autopct='%1.0f%%', startangle=90)
axes[0].set_title(rtl('התפלגות קטגוריות'), fontsize=12, fontweight='bold')

# לוח 2: ביטחון לפי מסמך + תוויות
for i, (_, row) in enumerate(df_viz.iterrows()):
    c = colors.get(row['קטגוריה'], '#888')
    axes[1].scatter(i, row['ציון_ביטחון'], color=c, s=140, alpha=0.9, zorder=3)
    axes[1].annotate(rtl(row['שם_קצר']),
                     xy=(i, row['ציון_ביטחון']),
                     xytext=(8, 5), textcoords='offset points',
                     fontsize=9, color='#222')
axes[1].axhline(y=0.7, color='orange', linestyle='--', alpha=0.7, label=rtl('סף 0.7'))
axes[1].set_ylim(0, 1.15)
axes[1].set_xlim(-0.5, len(df_viz) - 0.3)
axes[1].set_xticks([])
axes[1].set_ylabel(rtl('ציון ביטחון'), fontsize=11)
axes[1].set_title(rtl('ציון ביטחון הסיווג'), fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# לוח 3: ביטחון מול אורך + תוויות
for _, row in df_viz.iterrows():
    c = colors.get(row['קטגוריה'], '#888')
    axes[2].scatter(row['אורך_טקסט'], row['ציון_ביטחון'],
                    color=c, s=140, alpha=0.9, zorder=3)
    axes[2].annotate(rtl(row['שם_קצר']),
                     xy=(row['אורך_טקסט'], row['ציון_ביטחון']),
                     xytext=(6, 5), textcoords='offset points',
                     fontsize=9, color='#222')
axes[2].set_xlabel(rtl('אורך טקסט (מילים)'), fontsize=11)
axes[2].set_ylabel(rtl('ציון ביטחון'), fontsize=11)
axes[2].set_title(rtl('ביטחון לעומת אורך'), fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
from matplotlib.patches import Patch
axes[2].legend(handles=[Patch(facecolor=colors.get(c, '#888'), label=rtl(c))
               for c in df_viz['קטגוריה'].unique()], fontsize=9)

plt.suptitle(rtl('ניתוח תוצאות AI – סיווג טקסטים'), fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_ai_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 הגרף נשמר: 01_ai_results.png")


In [ ]:
# ============================================================
# שמירת הנתונים
# ============================================================

print("💾 שומר נתונים...")

if 'df_results' in dir():
    df_results.to_csv('ai_analysis_results.csv', index=False, encoding='utf-8-sig')
    print("  ✅ ai_analysis_results.csv")

print("\n🎉 מחברת 5 הושלמה!")
print()
print("📚 משאבים לשימוש ב-AI אמיתי:")
print("  -> Claude API: https://console.anthropic.com/")
print("  -> AlephBERT:  https://huggingface.co/onlplab/alephbert-base")
print("  -> Gemini API: https://ai.google.dev/")
print()
print("⚠️ תמיד:")
print("  1. בדקו עובדות שAI מספק")
print("  2. ציינו שימוש ב-AI בפרסומים")
print("  3. שמרו על שיפוט ביקורתי!")


## תרגילים

### תרגיל 1 – בסיסי
כתבו 3 prompts שונים לאותה משימה (סיכום טקסט היסטורי).  
השוו את התוצאות – מה ה-prompt שנתן את התוצאה הטובה ביותר?

### תרגיל 2 – בינוני
הוסיפו 10 טקסטים חדשים לנתוני האימון בפונקציית הסיווג.  
שפרו את הדיוק ומדדו שינוי בתוצאות ה-classification_report.

### תרגיל 3 – מתקדם
השיגו מפתח API חינמי (Claude/Gemini) והריצו ניתוח אמיתי  
על 20 ערכים מהקורפוס שנאסף במחברת 1.  
השוו: AI לעומת TF-IDF בסיווג – מי מדויק יותר?

---

## משאבים
- [Anthropic Claude API](https://docs.anthropic.com/)
- [OpenAI API](https://platform.openai.com/docs/)
- [AlephBERT](https://huggingface.co/onlplab/alephbert-base)
- Bender et al. (2021). *On the Dangers of Stochastic Parrots*. FAccT.
- [AI Ethics in Digital Humanities](https://dhdebates.gc.cuny.edu/)

---

## 🏠 משימת הבית

### מה להגיש:
1. **Prompt מקורי** – כתבו פרומפט לניתוח טקסט היסטורי / ארכיאולוגי משלכם
2. **תגובת Gemini** – הריצו את הפרומפט שלכם עם `call_gemini()` והדביקו את התגובה
3. **הערכה** – כתבו 2-3 משפטים: מה טוב בתגובה? מה הייתם משפרים בפרומפט?

### כיצד להגיש:
1. **File → Save a copy in Drive** – שמרו עותק אישי
2. **File → Download → Download .ipynb** – הורידו את הקובץ
3. העלו ל-Moodle

---

### 🤖 עצה: אם נתקעתם – שאלו את Gemini!
פתחו טאב חדש ב-[Google AI Studio](https://aistudio.google.com/) ושאלו:

```
אני יוצרת פרומפט לניתוח טקסט היסטורי בשיעור DH.
הפרומפט שלי הוא: "[הדביקי כאן]"
מה אפשר לשפר בו?
```
